In [1]:
import json
import csv
import os
from itertools import combinations

import polars as pl


def load_author_map(path: str) -> pl.DataFrame:
    #Load the author map JSON: {raw_name: normalized_name} into a long dfwith columns [raw, normalized]."""
    with open(path, "r", encoding="utf-8") as f:
        author_map = json.load(f)
    return pl.DataFrame(
        {"raw": list(author_map.keys()), "normalized": list(author_map.values())}
    )


def load_norton_csv(path: str) -> pl.DataFrame:
    #Load the Norton anthology CSV into a df with columns [raw, normalized],
    df = pl.read_csv(path, schema_overrides={"PublicationYear": pl.String, "ISBN": pl.String, "Checkouts": pl.String}
)
    df = (
        df.select(
            pl.col("Creator").cast(pl.Utf8).str.strip_chars().alias("raw"),
            pl.col("Normalized.Creator").cast(pl.Utf8).str.strip_chars().alias("normalized"),
        )
        .filter((pl.col("raw") != "") & (pl.col("normalized") != ""))
        .filter(pl.col("raw").is_not_null() & pl.col("normalized").is_not_null())
    )
    return df


# appl

def build_groups(df: pl.DataFrame) -> pl.DataFrame:
    #Returns df with columns [normalized, raw_names (list[str])].
    return df.group_by("normalized").agg(pl.col("raw").unique().alias("raw_names"))


def make_pairs(raw_names: list[str]) -> list[list[str]]:
    #All unordered pairs of raw names within a group, each pair sorted so (a, b) and (b, a) merge
    uniq = sorted(set(raw_names))
    return [list(p) for p in combinations(uniq, 2)]


def build_pairs_df(groups_df: pl.DataFrame) -> pl.DataFrame:

    #Expand each group's raw_names list into all pairwise combinations.
    #Returns df [normalized, pair] with one row per pair (pair is a 2-elem list).
    with_pairs = groups_df.with_columns(
        pl.col("raw_names")
        .map_elements(make_pairs, return_dtype=pl.List(pl.List(pl.Utf8)))
        .alias("pairs")
    )
    exploded = with_pairs.explode("pairs").filter(pl.col("pairs").is_not_null())
    return exploded.select("normalized", pl.col("pairs").alias("pair"))


def pairs_df_to_list(pairs_df: pl.DataFrame) -> list[tuple[str, str]]:
    #Flatten a [normalized, pair] df down to a plain python list of (raw_a, raw_b) tuples (dropping normalized)
    return [tuple(p) for p in pairs_df["pair"].to_list()]


def build_pair_lookup(pairs_df: pl.DataFrame) -> dict[tuple[str, str], list[str]]:
    #Map each pair to the sorted list of normalized names it came from for output rows later
    lookup: dict[tuple[str, str], set[str]] = {}
    for normalized, pair in zip(pairs_df["normalized"].to_list(), pairs_df["pair"].to_list()):
        lookup.setdefault(tuple(pair), set()).add(normalized)
    return {pair: sorted(normalizeds) for pair, normalizeds in lookup.items()}



def compare_pairs(
    am_pairs: list[tuple[str, str]],
    norton_pairs: list[tuple[str, str]],
    norton_unique_names: set[str],
) -> tuple[set, set, set, list]:
    
    #Restrict am_pairs to only pairs whose both raw names are known to Norton then compare 
    #Returns: (correct_pairs, false_positive_pairs, false_negative_pairs, am_pairs_subset)
    
    am_pairs_subset = [
        p for p in am_pairs if p[0] in norton_unique_names and p[1] in norton_unique_names
    ]

    am_set = set(am_pairs_subset)
    norton_set = set(norton_pairs)

    correct_pairs = am_set & norton_set
    false_positive_pairs = am_set - norton_set   # author map thinks it's a match, Norton disagrees
    false_negative_pairs = norton_set - am_set   # Norton thinks it's a match, author map missed it

    return correct_pairs, false_positive_pairs, false_negative_pairs, am_pairs_subset


def rows_from_pairs(
    pairs: set[tuple[str, str]],
    category: str,
    reason: str,
    am_lookup: dict[tuple[str, str], list[str]],
    norton_lookup: dict[tuple[str, str], list[str]],
) -> list[dict]:
    rows = []
    for a, b in sorted(pairs):
        rows.append({
            "raw_name_a": a,
            "raw_name_b": b,
            "author_map_normalized": "; ".join(am_lookup.get((a, b), [])),
            "norton_normalized": "; ".join(norton_lookup.get((a, b), [])),
            "category": category,
            "reason": reason,
        })
    return rows


def save_results(
    correct: list[dict],
    false_pos: list[dict],
    false_neg: list[dict],
    output_dir: str,
):
    #Save results to CSV files in the output directory.
    fieldnames = [
        "raw_name_a",
        "raw_name_b",
        "author_map_normalized",
        "norton_normalized",
        "category",
        "reason",
    ]

    files = {
        "correct": (correct, "author_correct_pairs.csv"),
        "fp": (false_pos, "author_false_positives.csv"),
        "fn": (false_neg, "author_false_negatives.csv"),
    }

    for key, (data, filename) in files.items():
        path = os.path.join(output_dir, filename)
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data)
        print(f"  Saved {len(data):>5} {key:>12} rows -> {path}")


In [2]:

author_map_path = "../../data-prep/author_map.json"
norton_csv_path = "norton-anthology_spl-checkouts_2005-2025(1).csv"

am_df = load_author_map(author_map_path)

norton_df = load_norton_csv(norton_csv_path)

# Groupby normalized to unique raw names, for each source
am_groups_df = build_groups(am_df)
norton_groups_df = build_groups(norton_df)
print(am_groups_df)
print(norton_groups_df)

# Expand each group into every pairwise combination of its raw names
am_pairs_df = build_pairs_df(am_groups_df)
norton_pairs_df = build_pairs_df(norton_groups_df)

print(am_pairs_df)

am_pairs = pairs_df_to_list(am_pairs_df)
norton_pairs = pairs_df_to_list(norton_pairs_df)
am_lookup = build_pair_lookup(am_pairs_df)
norton_lookup = build_pair_lookup(norton_pairs_df)

print(am_pairs)
print(norton_pairs)

# Compare the two plain python lists of pairs (author map pairs are first
# restricted to only the raw names that actually appear in Norton, since
# that's the only slice we can evaluate against)

norton_unique_names = set(norton_df["raw"].unique().to_list())
correct_pairs, fp_pairs, fn_pairs, am_pairs_subset = compare_pairs(
    am_pairs, norton_pairs, norton_unique_names
)



correct = rows_from_pairs(correct_pairs, "correct_pair", "", am_lookup, norton_lookup)
false_pos = rows_from_pairs(
    fp_pairs, "false_positive", "Author map pairs these but Norton does not", am_lookup, norton_lookup
)

print(correct)
false_neg = rows_from_pairs(
    fn_pairs, "false_negative", "Norton pairs these but author map does not", am_lookup, norton_lookup
)
print(false_neg)


output_dir = os.path.dirname(os.path.abspath(norton_csv_path))
save_results(correct, false_pos, false_neg, output_dir)

print("\nDone.")



shape: (344_604, 2)
┌─────────────────────────────────┬─────────────────────────────────┐
│ normalized                      ┆ raw_names                       │
│ ---                             ┆ ---                             │
│ str                             ┆ list[str]                       │
╞═════════════════════════════════╪═════════════════════════════════╡
│ liz lawrence                    ┆ ["Liz Lawrence"]                │
│ jennifer park, 1981             ┆ ["Park, Jennifer, 1981-", "Jen… │
│ dave liniger                    ┆ ["Dave Liniger", "Liniger, Dav… │
│ dan burne jones                 ┆ ["Jones, Dan Burne"]            │
│ sherman robley cook             ┆ ["Cook, Sherman Robley."]       │
│ …                               ┆ …                               │
│ benjamin l corey                ┆ ["Corey, Benjamin L."]          │
│ pamela grundy                   ┆ ["Grundy, Pamela"]              │
│ thomas read rootes cobb, 1823   ┆ ["Cobb, Thomas Read Rootes, 18… │


In [3]:
print(f"Recall", len(correct) / (len(correct) + len(false_neg)))
print(f"Precision", len(correct) / (len(correct) + len(false_pos)))

Recall 0.9279279279279279
Precision 1.0


In [7]:

# --- Baseline: pure fuzzy matching on raw creator strings, no author map involved ---
#
# Raw Creator values mix "First Last" and "Last, First, birth-death" formats, plus
# inconsistent trailing commas/periods and life-date tokens across variants of the
# same person (e.g. "Ashbery, John, 1927-" vs "Ashbery, John, 1927-2017,"). We use
# rapidfuzz's token_set_ratio because it's order-independent (handles the name-order
# swap) and tolerant of extra/missing tokens (handles the differing life-dates and
# punctuation) -- properties plain edit-distance ratios don't have.

from rapidfuzz import fuzz, process, utils


def fuzzy_similarity_matrix(names: list[str]):
    #Full pairwise token_set_ratio similarity matrix (0-100), computed in one C call
    return process.cdist(names, names, scorer=fuzz.token_set_ratio, processor=utils.default_process)


def sweep_thresholds(
    names: list[str],
    sim_matrix,
    gold_pairs: set[tuple[str, str]],
    thresholds: list[float],
) -> pl.DataFrame:
    #For each threshold, treat name pairs scoring at or above it as "same author" and score against gold_pairs
    n = len(names)
    rows = []
    for t in thresholds:
        fuzzy_set = {
            tuple(sorted((names[i], names[j])))
            for i in range(n)
            for j in range(i + 1, n)
            if sim_matrix[i, j] >= t
        }
        correct = fuzzy_set & gold_pairs
        fp = fuzzy_set - gold_pairs
        fn = gold_pairs - fuzzy_set
        rows.append({
            "threshold": t,
            "predicted_pairs": len(fuzzy_set),
            "correct": len(correct),
            "false_positives": len(fp),
            "false_negatives": len(fn),
            "precision": len(correct) / len(fuzzy_set) if fuzzy_set else float("nan"),
            "recall": len(correct) / len(gold_pairs) if gold_pairs else float("nan"),
        })
    return pl.DataFrame(rows)


In [8]:

# Run the fuzzy baseline against exactly the same gold pairs (norton_pairs) the author map was scored against
names = sorted(norton_unique_names)
sim_matrix = fuzzy_similarity_matrix(names)
norton_pairs_set = set(norton_pairs)

thresholds = [60, 65, 70, 75, 80, 85, 90, 95, 98]
fuzzy_results_df = sweep_thresholds(names, sim_matrix, norton_pairs_set, thresholds)

with pl.Config(tbl_rows=-1):
    print(fuzzy_results_df)

print(f"\nFor comparison, author map: recall={len(correct) / (len(correct) + len(false_neg)):.4f}, "
      f"precision={len(correct) / (len(correct) + len(false_pos)):.4f}")


shape: (9, 7)
┌───────────┬─────────────────┬─────────┬─────────────────┬─────────────────┬───────────┬──────────┐
│ threshold ┆ predicted_pairs ┆ correct ┆ false_positives ┆ false_negatives ┆ precision ┆ recall   │
│ ---       ┆ ---             ┆ ---     ┆ ---             ┆ ---             ┆ ---       ┆ ---      │
│ i64       ┆ i64             ┆ i64     ┆ i64             ┆ i64             ┆ f64       ┆ f64      │
╞═══════════╪═════════════════╪═════════╪═════════════════╪═════════════════╪═══════════╪══════════╡
│ 60        ┆ 277             ┆ 219     ┆ 58              ┆ 3               ┆ 0.790614  ┆ 0.986486 │
│ 65        ┆ 246             ┆ 219     ┆ 27              ┆ 3               ┆ 0.890244  ┆ 0.986486 │
│ 70        ┆ 229             ┆ 219     ┆ 10              ┆ 3               ┆ 0.956332  ┆ 0.986486 │
│ 75        ┆ 221             ┆ 219     ┆ 2               ┆ 3               ┆ 0.99095   ┆ 0.986486 │
│ 80        ┆ 218             ┆ 216     ┆ 2               ┆ 6               ┆